In [ ]:
import sys
# sys.path.append("/Users/bubble/Desktop/Project/Infrasound Sensor/Layout/Code/acousticsensor")

import gdsfactory as gf
cell_temp = gf.Component()

In [ ]:
# square mesh
L = [1000]
b = [500]
g = [48]
w = 2

beamNum = []
side =[None] * 4
k = (len(L) * len(g))
mesh = [None] * k
for i in range(k):
    mesh[i] = gf.Component()
for i in range(len(side)):
    side[i] = gf.Component()
count = 0
origin = (5000 / 3, 0)

for i in range(len(side)):
    if i <2:
        for j in range(len(g)):
            # single grid
            row =   L[i] // (g[j]+w)
            column = row if row % 2 == 0 else row + 1
            beamNum.append((row+1)+ column)
            height = L[i] / row - w
            sg = gf.components.rectangle(size=(g[j], height), layer=(8, 0))
            for m in range(column):
                for n in range(row):
                    sf_ref = mesh[count] << sg
                    sf_ref.move((origin[0]+m*(g[j]+w), origin[1]+n*(height+w)))
            # frame
            fra_width = column * (g[j] + w) + w
            frame = gf.components.rectangle(size=(fra_width, L[i]), layer=(9, 0))
            frame_ref = mesh[count] << frame
            frame_ref.movex(origin[0]-w)
            # crystal
            mid = origin[0] - w + fra_width / 2
            cry_size = 50
            # cry_height = height+0.1 if j != 0 else 2*height+w+0.1
            crystal = gf.components.rectangle(size=(cry_size, cry_size), layer=(10, 0))
            crystal_ref = mesh[count] << crystal
            crystal_ref.move((mid-cry_size/2, L[i]-cry_size-w+0.05))
            # deposition area
            gold_size = cry_size - 6
            gold = gf.components.rectangle(size=(gold_size, gold_size+2), layer=(11, 0))
            gold_ref = mesh[count] << gold
            gold_ref.move((mid-gold_size/2, L[i]-gold_size-w-3))
            # move the mesh
            temp = count // 2
            mesh_ref = side[temp] << mesh[count]
            mesh_ref.movex((count%2) * 5000 / 3 - L[i] / 2) 
            count += 1
            # length mark
            T = gf.components.text(f"L={L[i]} gap={g[i]}", size=50, layer=(1, 0))
            T_ref = side[i] << T
            T_ref.move(((count%2+1) * 5000/3 - L[i]/2 + 200, -200))
    else:
        # add some beams
        L_beams = [1000]*3
        start = 1000
        interval = (5000 - 2*start) / (len(L_beams)+1)
        for k in range(len(L_beams)):
            beam = gf.components.rectangle(size=(w, L_beams[k]), layer=(9, 0))
            beam_ref = side[i] << beam
            beam_ref.move((start + (k+1)*interval, 0))
        # length mark
        T = gf.components.text(f"L={L_beams[0]} b={w}", size=50, layer=(1, 0))
        T_ref = side[i] << T
        T_ref.move((5000/2-200, -200))

# the whole chip
block = gf.Component()
for i in range(len(side)):
    # meshes: side[0] and side[1]
    # cantilevers: side[2] and side[3]
    side_ref = block << side[i]
    if i == 0:
        pass
    elif i == 1:
        side_ref.dmirror_y(5000/2).dmirror_x(5000/2)
    elif i == 2:
        side_ref.drotate(angle=-90, center=(5000/2, 5000/2))
    else:
        side_ref.drotate(angle=90, center=(5000/2, 5000/2))

block.show()
# print(beamNum)

